# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [55]:
import os
import time
import re
import requests
import schedule
import json
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [56]:
def get_book_data(book_url: str) -> dict:
    """
    Собирает данные о книге со страницы товара.

    Args:
        book_url (str): ссылка на страницу книги.

    Returns:
        dict: словарь с полями title, category, price, rating,
            availability (число штук в наличии), description, url, а также
            все характеристики из таблицы Product Information.
    """
    response = requests.get(book_url, timeout=10)
    response.raise_for_status()

    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')
    product = soup.find('article', class_='product_page')

    title = product.find('h1').get_text(strip=True)
    price = product.find('p', class_='price_color').get_text(strip=True)

    rating_class = product.find('p', class_='star-rating')['class'][1]
    rating = ['One', 'Two', 'Three', 'Four', 'Five'].index(rating_class) + 1

    availability_text = product.find(
        'p', class_='availability').get_text(strip=True)
    match = re.search(r'\((\d+) available\)', availability_text)
    availability = int(match.group(1)) if match else 0

    description = ''
    description_block = product.find('div', id='product_description')
    if description_block is not None:
        description = description_block.find_next_sibling(
            'p').get_text(strip=True)

    category = soup.select('ul.breadcrumb li')[2].get_text(strip=True)

    data = {
        'title': title,
        'category': category,
        'price': price,
        'rating': rating,
        'availability': availability,
        'description': description,
        'url': book_url,
    }

    for row in product.select('table.table-striped tr'):
        data[row.find('th').get_text(strip=True)] = row.find(
            'td').get_text(strip=True)

    return data

In [ ]:
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'category': 'Poetry',
 'price': '£51.77',
 'rating': 3,
 'availability': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place 

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [58]:
def scrape_books(is_save: bool = True, max_pages: int = None) -> list:
    """Обходит все страницы каталога и собирает данные о каждой книге.

    Args:
        is_save (bool): если True, результат сохраняется в
            artifacts/books_data.txt (одна книга — одна JSON-строка).
        max_pages (int | None): ограничение числа страниц для отладки; None —
            обойти весь каталог.

    Returns:
        list[dict]: список словарей с данными о книгах.
    """
    books = []
    page = 1
    while max_pages is None or page <= max_pages:
        response = requests.get(
            f'http://books.toscrape.com/catalogue/page-{page}.html', timeout=10
        )
        if response.status_code == 404:
            break
        response.raise_for_status()
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')

        for link in soup.select('article.product_pod h3 a'):
            book_url = 'http://books.toscrape.com/catalogue/' + link['href']
            books.append(get_book_data(book_url))
        page += 1

    if is_save:
        if os.path.basename(os.getcwd()) == 'notebooks':
            output_dir = os.path.join('..', 'artifacts')
        else:
            output_dir = 'artifacts'
            
        os.makedirs(output_dir, exist_ok=True)
        file_path = os.path.join(output_dir, 'books_data.txt')
        with open(file_path, 'w', encoding='utf-8') as file:
            for book in books:
                file.write(json.dumps(book, ensure_ascii=False) + '\n')

    return books

In [59]:
res = scrape_books(is_save=True, max_pages=2)
print(type(res), len(res))

<class 'list'> 40


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [60]:
def run_scheduler(run_time: str = '19:00', max_seconds: int = None) -> None:
  """Запускает сбор данных о книгах каждый день в заданное время.

  Args:
      run_time (str): время запуска в формате 'ЧЧ:ММ'.
      max_seconds (int | None): через сколько секунд остановить цикл (для
          проверки); None — работать бесконечно.
  """
  schedule.clear()
  schedule.every().day.at(run_time).do(
      scrape_books, is_save=True, max_pages=1
  )

  start = time.time()
  while max_seconds is None or time.time() - start < max_seconds:
    schedule.run_pending()
    time.sleep(10) 


if __name__ == '__main__':
  print('Запуск планировщика в тестовом режиме на 60 секунд...')
  current_time_str = (datetime.now() + timedelta(minutes=1)).strftime('%H:%M')
  run_scheduler(current_time_str, max_seconds=60)
  print('Тест планировщика успешно завершен!')

Запуск планировщика в тестовом режиме на 60 секунд...
Тест планировщика успешно завершен!


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [64]:
! pytest ../tests/test_scraper.py

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.2.0, pluggy-1.7.0
rootdir: c:\Users\ДолговаЕВ\Desktop\МФТИ\Python\HW_03_python_ds
plugins: platformdirs-4.14.5
collected 5 items

..\tests\test_scraper.py .....                                           [100%]

============================== 5 passed in 1.26s ==============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Необходимо было сделать парсер сайта Books to Scrape, написать функции get_book_data и scrape_books, ежедневный запуск в 19:00 через schedule, тесты на pytest, а также оформить проекта для GitHub. В результате получился проект с функциями get_book_data, scrape_books и run_scheduler, пятью тестами, размещённый на GitHub`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Сразу получилось всё кроме функции run_scheduler, с ней в итоге сама разобралась и путь неправильно создала папку artifacts в функции scrape_books, тут ИИ помог`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Использовала для настройки окружения и создания папки artifacts, забыла про команду os.makedirs()`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Вроде бы со всем разобралась`

**5. Самооценка по критериям задания**:

`Ответ: Вроде бы всё получилось, поэтому, наверное, можно сказать, что я справилась`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Все задачи считаю полезными, они помогли ещё раз закрепить пройденный материал`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

